# Visual search — full index build (Colab)

Run this on a Colab GPU runtime to build `embeddings.bin` + `manifest.json` for the full ~250k object catalogue.

Workflow:
1. Switch the runtime to GPU (`Runtime → Change runtime type → T4` or A100).
2. Run all cells in order.
3. When prompted, upload **two files**: `build_index.py` and a `feed.jsonl` produced locally by `node scripts/visual-search-build/visual-search-feed.mjs`.
4. The build runs (~5–15 min on T4 once HTTP fetches catch up; HTTP usually dominates).
5. The final cell downloads `embeddings.bin` and `manifest.json` to your Mac. Upload those to `coimages.../models/` manually.

See `README.md` in this folder for the full pipeline. Internal decisions and validation numbers are deliberately kept out of this public repo — ask Jamie if you need that context.

## 1. Verify GPU runtime

In [ ]:
!nvidia-smi

## 2. Install dependencies

Colab ships with `torch`, `numpy`, `pillow`, `tqdm` pre-installed. We only need `open_clip_torch` and `httpx`.

In [ ]:
!pip -q install open_clip_torch httpx

## 3. Upload `build_index.py` and `feed.jsonl`

The Python script is the same one that lives in the repo at `scripts/visual-search-build/build_index.py`. The JSONL is whatever you produced locally with `node scripts/visual-search-build/visual-search-feed.mjs`.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    print(f'  uploaded {name} ({len(data):,} bytes)')
assert 'build_index.py' in uploaded, 'Please upload build_index.py'
assert 'feed.jsonl' in uploaded, 'Please upload feed.jsonl'

## 4. Run the build

T4 is enough; A100 is faster but rarely the bottleneck (HTTP fetch usually dominates on the first run). Subsequent runs hit the disk cache and complete in minutes.

Set `--limit` to a small number (e.g. 200) for a smoke test before committing to the full run.

In [ ]:
!python build_index.py \
    --input feed.jsonl \
    --output-dir ./out \
    --device cuda \
    --batch-size 128 \
    --concurrency 24

## 5. Verify outputs

In [ ]:
import json, os, numpy as np
m = json.load(open('out/manifest.json'))
e = np.fromfile('out/embeddings.bin', dtype=np.float32).reshape(-1, m['embed_dim'])
assert e.shape[0] == m['count'], (e.shape, m['count'])
norms = np.linalg.norm(e, axis=1)
bin_bytes = os.path.getsize('out/embeddings.bin')
print(f'count        : {m["count"]:,}')
print(f'embed_dim    : {m["embed_dim"]}')
print(f'model        : {m["model"]} / {m["model_revision"]}')
print(f'built_at     : {m["built_at"]}')
print(f'norm range   : {norms.min():.4f} – {norms.max():.4f}  (expect ~1.0)')
print(f'bin size     : {bin_bytes:,} bytes')

## 6. Download to your Mac

Both files at once. Upload them manually to `coimages.../models/` afterwards.

In [ ]:
from google.colab import files
files.download('out/embeddings.bin')
files.download('out/manifest.json')